# Masking delle feature FC in funzione della lesione

## Il problema, in parole semplici

Ogni paziente stroke ha due tipi di dati sul cervello:

- **La lesione**: la zona di tessuto danneggiato dall'ictus (un'immagine 3D che dice, voxel per voxel, "qui c'e' danno / qui no").
- **La connettivita' funzionale (FC)**: quanto "comunicano tra loro" le diverse zone del cervello a riposo, misurata con la risonanza funzionale. Il cervello viene diviso in centinaia di piccole zone ("nodi" o "parcel"), e per ogni coppia di zone si misura quanto la loro attivita' e' sincronizzata nel tempo. Il risultato e' una grande tabella nodo x nodo (una matrice), gia' calcolata e disponibile per i nostri pazienti WashU.

**Il problema**: se una zona e' dentro la lesione (tessuto morto/danneggiato), il segnale che ne esce non e' piu' "attivita' cerebrale reale" - e' rumore o assenza di segnale. Se non lo togliamo, rischiamo di scambiare "questa zona e' distrutta" per "comunicazione alterata tra zone sane". Serve quindi **azzerare** ogni valore di connettivita' che coinvolge una zona troppo danneggiata, prima di poter usare questi dati per collegare lesione/disconnessione ai deficit clinici (linguaggio, attenzione, motricita'...).

Questo notebook fa esattamente questo, un soggetto alla volta, e verifica il risultato su dati reali (non simulati) prima di trasformarlo in codice definitivo.

## Da dove viene il metodo

Non e' un metodo inventato per l'occasione - e' lo stesso approccio gia' usato in letteratura su questa stessa coorte di pazienti (WashU, laboratorio Corbetta):

- **Siegel et al. 2016, *PNAS*** — per ogni paziente, le connessioni delle zone dentro la lesione vengono azzerate.
- **Griffis et al. 2019, *Neuron*** (gia' salvato in `assets/papers/`) — versione piu' precisa: una zona viene esclusa solo se **una percentuale sufficiente** del suo territorio e' dentro la lesione (non basta un solo punto danneggiato). La soglia standard usata in questo campo e' **il 50%**: se piu' della meta' di una zona e' lesionata, la zona viene considerata inutilizzabile.
- **XCP-D** (`xcp_d/interfaces/connectivity.py`, classe `NiftiParcellate`) — il software che ha effettivamente generato le nostre matrici FC ha gia' un meccanismo identico (`min_coverage`, soglia di default 0.5), ma per un problema diverso (zone scoperte dalla risonanza). Qui riusiamo lo stesso meccanismo, con la lesione al posto del problema di copertura originale.

**Decisione presa con l'utente**: non scrivere una funzione di conteggio a mano - riusare uno strumento gia' pronto e collaudato (`nilearn`, una libreria standard per l'analisi di immagini cerebrali), con lo stesso schema che usa XCP-D.

In [1]:
import warnings

# nilearn/nibabel emettono warning di deprecazione non rilevanti per questa analisi - silenziati
# solo per leggibilita' dell'output, non nascondono errori reali (quelli restano ValueError/AssertionError).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import glob  # per trovare il file CSV della matrice FC senza scrivere il nome esatto a mano

import nibabel as nib  # legge/scrive immagini cerebrali in formato NIfTI (.nii.gz)
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img  # riallinea due immagini 3D sulla stessa griglia spaziale
from nilearn.maskers import NiftiLabelsMasker  # estrae valori per-zona da un'immagine, data una mappa di zone

# --- Parametri di questa run: un soggetto, una versione della mappa cerebrale ---
SUBJECT = "sub-STUNIPD0003"  # paziente scelto per il test: ha una lesione abbastanza grande da mostrare l'effetto
ATLAS_COMBO = "atlas-Yan200TianS2Buckner7N"  # una delle 12 mappe cerebrali disponibili (200 zone corticali + subcortex + cervelletto)
MIN_COVERAGE = 0.5  # soglia: sotto il 50% di territorio sano, la zona e' "compromessa" (standard di campo, vedi sopra)

DATA_ROOT = "../data/clinical_connectome/derivatives/UNIPD/WashU"  # dati del paziente (lesione + connettivita')
ATLAS_ROOT = f"../assets/atlases/fmriprep/{ATLAS_COMBO}"  # mappa cerebrale di riferimento (copiata dal server)

## 1. La mappa del cervello ("atlante")

Per sapere "quanto e' dentro la lesione" ogni singola zona, serve prima una **mappa di riferimento** che dice esattamente dove sono i confini di ciascuna delle ~240 zone usate per calcolare la connettivita'. Senza questa mappa non sappiamo a quale zona appartiene ogni punto del cervello.

**Cosa contiene questa mappa** (gia' pronta sul server EBRAIN, cartella `Atlases/fmriprep/atlas-Yan200TianS2Buckner7N/`, copiata in locale):
- Un'immagine 3D (`*_res-2_dseg.nii.gz`) dove ogni punto del cervello ha un numero: quel numero identifica a quale delle ~240 zone appartiene quel punto (0 = fuori dal cervello).
- Una tabella (`*_dseg.tsv`) che traduce ogni numero in un nome leggibile (es. "corteccia visiva sinistra").

Usiamo direttamente la versione a **2mm** di risoluzione (`res-2`) perche' e' gia' alla stessa risoluzione della lesion mask WashU — evita un passaggio di ricampionamento in piu'.

In [2]:
# squeeze_image: il file ha una dimensione extra inutile (4 dimensioni invece di 3), la togliamo
atlas_img = nib.squeeze_image(
    nib.load(f"{ATLAS_ROOT}/{ATLAS_COMBO}_space-MNI152NLin6Asym_res-2_dseg.nii.gz")
)

# la tabella che traduce ogni numero-zona nel suo nome (es. 1 -> "7Networks_LH_Default_IPL_1")
label_table = pd.read_csv(f"{ATLAS_ROOT}/{ATLAS_COMBO}_dseg.tsv", sep="\t")
label_ids = label_table["index"].tolist()  # tutti i numeri-zona attesi, nell'ordine ufficiale
id_to_name = dict(zip(label_table["index"], label_table["label"]))  # dizionario numero -> nome

print(f"Atlante caricato: {len(label_ids)} nodi")
label_table.head(3)  # anteprima: come si presenta la tabella

Atlante caricato: 239 nodi


,index,label
0,1,7Networks_LH_Default_IPL_1
1,2,7Networks_LH_Default_IPL_2
2,3,7Networks_LH_Default_IPL_3


## 2. Allineare la lesione alla mappa

La lesion mask del paziente e la mappa cerebrale sono due immagini 3D separate, create in momenti/contesti diversi. Anche se hanno la **stessa dimensione** (91 x 109 x 91 punti), questo non garantisce che il punto "numero 50" nella prima immagine corrisponda allo stesso punto fisico del cervello nella seconda: dipende da come ciascuna immagine e' orientata nello spazio (registrata nel suo header/"affine"). Qui sotto infatti l'asse sinistra-destra ha il segno invertito tra le due immagini.

**Perche' e' importante controllarlo**: se ignorassimo questo dettaglio e allineassimo le due immagini "punto per punto" senza correggere l'orientamento, rischieremmo un errore silenzioso e grave — una lesione dell'emisfero sinistro verrebbe scambiata per una dell'emisfero destro (flip sinistra/destra). Per questo si usa sempre una funzione di libreria (`resample_to_img`) che legge l'orientamento vero di entrambe le immagini e le riallinea correttamente, invece di assumere che coincidano.

In [3]:
lesion_path = (
    f"{DATA_ROOT}/manual_masks/{SUBJECT}/anat/"
    f"{SUBJECT}_space-MNI152NLin6Asym_label-lesion_mask.nii.gz"
)
lesion_img = nib.load(lesion_path)  # la lesione di questo paziente, cosi' come e' stata salvata

print("Lesion mask affine (WashU):\n", lesion_img.affine)
print("Atlas affine (server, res-2):\n", atlas_img.affine)
print("-> stessa shape, segno asse X diverso: resample via affine obbligatorio.")

# resample_to_img: prende la lesione e la "riproietta" esattamente sulla griglia della mappa cerebrale,
# usando le informazioni di orientamento reali di entrambe (non le posizioni grezze dei punti).
# interpolation="nearest": la lesione e' binaria (0/1) - non ha senso creare valori intermedi come 0.3,
# quindi ogni nuovo punto prende il valore del punto originale piu' vicino, non una media.
lesion_resampled = resample_to_img(
    lesion_img, atlas_img, interpolation="nearest", force_resample=True, copy_header=True
)
# > 0.5: dopo il riallineamento alcuni valori potrebbero non essere piu' esattamente 0 o 1 - li
# riportiamo a un'immagine strettamente binaria (1 = lesionato, 0 = sano).
lesion_data = (np.asarray(lesion_resampled.get_fdata()) > 0.5).astype(np.int32)
print(f"\nVoxel lesionati ({SUBJECT}, griglia atlante 2mm): {int(lesion_data.sum())}")

Lesion mask affine (WashU):
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
Atlas affine (server, res-2):
 [[   2.    0.    0.  -90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
-> stessa shape, segno asse X diverso: resample via affine obbligatorio.

Voxel lesionati (sub-STUNIPD0003, griglia atlante 2mm): 7042


## 3. Quanto e' danneggiata ciascuna zona?

Ora che lesione e mappa sono sulla stessa griglia, per ogni zona del cervello calcoliamo:

```
% di zona sana = (punti sani in quella zona) / (punti totali in quella zona)
```

Se questo valore scende sotto il 50%, la zona e' "compromessa" e verra' azzerata nella matrice di connettivita'.

**Come lo calcoliamo in pratica**: usiamo due volte lo stesso strumento (`NiftiLabelsMasker`, dalla libreria `nilearn`) — una volta per contare *tutti* i punti di ciascuna zona, una volta per contare solo i punti *sani* di ciascuna zona (dandogli in pasto una "maschera" che copre solo le parti non lesionate). Il rapporto tra i due conteggi e' la percentuale di zona sana. E' lo stesso identico principio usato da XCP-D per il suo controllo di qualita' (`min_coverage`).

### Due errori nascosti trovati testando su dati veri

Prima di fidarci del risultato, abbiamo testato questo calcolo su un paziente reale e confrontato i numeri con un conteggio manuale di controllo. Sono emersi due problemi non ovvi, che uno strumento "pronto all'uso" non segnala con nessun messaggio di errore:

1. **Un "trabocco" nei numeri (overflow)**: per contare i punti di ogni zona, il conteggio veniva fatto usando una casella di memoria capace di contenere solo numeri fino a 255 (`uint8`) — pensa a un contachilometri che torna a 0 dopo 256 invece di continuare a salire. Una zona con 695 punti veniva quindi contata come 183 (`695 - 256 - 256 = 183`), senza nessun avviso. **Fix**: usare una casella capace di numeri molto piu' grandi (`int32`).
2. **Una zona completamente distrutta "sparisce" invece di risultare 0% sana**: se una zona non ha piu' nemmeno un punto sano, lo strumento la **toglie del tutto** dal risultato, invece di dire "questa e' compromessa al 100%". Se non gestito esplicitamente, questo puo' far bloccare il programma (i due conteggi hanno lunghezze diverse e non si possono piu' confrontare posizione per posizione), oppure — molto peggio — far scivolare per errore il risultato di una zona su un'altra zona diversa. **Fix**: abbiniamo sempre i due conteggi per **nome/numero della zona**, mai per posizione nella lista, e trattiamo esplicitamente "zona sparita" come "0% sana".

In [4]:
def compute_parcel_coverage(atlas_img, label_ids, healthy_img):
    """Frazione di voxel sani per ciascuna zona, nello stesso ordine di label_ids.

    Una zona con zero voxel sani rimasti viene rimossa da NiftiLabelsMasker quando gli si
    passa mask_img (non restituita come 0) - gestito qui esplicitamente come coverage
    0.0, mai lasciato disallineare silenziosamente il conteggio mascherato da quello totale.
    """
    # Immagine "contatore": vale 1 in ogni punto del cervello. Sommandola zona per zona
    # otteniamo semplicemente "quanti punti ha quella zona". int32, non uint8: con una
    # casella troppo stretta il conteggio va in overflow silenzioso (bug verificato sopra).
    ones_img = nib.Nifti1Image(np.ones(atlas_img.shape, dtype=np.int32), atlas_img.affine, atlas_img.header)

    # Primo conteggio: quanti punti ha CIASCUNA zona in totale (nessuna maschera applicata).
    masker_total = NiftiLabelsMasker(labels_img=atlas_img, background_label=0, strategy="sum", standardize=False)
    n_total = np.squeeze(masker_total.fit_transform(ones_img))
    # labels_[1:]: il primo elemento e' sempre un segnaposto "Background" (zona 0, fuori dal cervello) - lo saltiamo.
    total_by_label = dict(zip(masker_total.labels_[1:], n_total))

    # Secondo conteggio: quanti punti SANI ha ciascuna zona (mask_img=healthy_img esclude i punti lesionati).
    masker_healthy = NiftiLabelsMasker(
        labels_img=atlas_img, mask_img=healthy_img, background_label=0, strategy="sum", standardize=False
    )
    n_healthy = np.squeeze(masker_healthy.fit_transform(ones_img))
    healthy_by_label = dict(zip(masker_healthy.labels_[1:], np.atleast_1d(n_healthy)))

    # Percentuale sana = punti sani / punti totali, per ciascuna zona attesa dalla tabella ufficiale.
    # .get(lbl, 0.0): se una zona e' sparita dal secondo conteggio (100% lesionata), il suo valore e' 0.0 - mai un errore.
    return np.array([healthy_by_label.get(lbl, 0.0) / total_by_label[lbl] for lbl in label_ids])


# healthy_img: l'"inverso" della lesione - vale 1 dove il cervello e' sano, 0 dove e' lesionato.
healthy_img = nib.Nifti1Image((1 - lesion_data), atlas_img.affine, atlas_img.header)
parcel_coverage = compute_parcel_coverage(atlas_img, label_ids, healthy_img)
compromised = parcel_coverage < MIN_COVERAGE  # True/False per ciascuna zona: e' sotto la soglia del 50%?
node_names = np.array([id_to_name[i] for i in label_ids])  # nomi leggibili delle zone, stesso ordine

print(f"Nodi totali: {len(node_names)} | compromessi (coverage sana < {MIN_COVERAGE}): {int(compromised.sum())}\n")
for name, cov in sorted(zip(node_names[compromised], parcel_coverage[compromised]), key=lambda x: x[1]):
    print(f"  {name}: coverage sana={cov:.3f} (overlap lesione={1 - cov:.1%})")

Nodi totali: 239 | compromessi (coverage sana < 0.5): 8

  Tian_pPUT-lh: coverage sana=0.000 (overlap lesione=100.0%)
  Tian_aGP-lh: coverage sana=0.019 (overlap lesione=98.1%)
  Tian_pGP-lh: coverage sana=0.067 (overlap lesione=93.3%)
  Tian_pCAU-lh: coverage sana=0.128 (overlap lesione=87.2%)
  Tian_aPUT-lh: coverage sana=0.295 (overlap lesione=70.5%)
  Tian_aCAU-lh: coverage sana=0.463 (overlap lesione=53.7%)
  7Networks_LH_SomMot_Ins_2: coverage sana=0.466 (overlap lesione=53.4%)
  7Networks_LH_SalVentAttn_Ins_1: coverage sana=0.499 (overlap lesione=50.1%)


**Il risultato ha senso clinicamente**: `sub-STUNIPD0003` ha una lesione dei gangli della base sinistri (putamen, pallido, caudato — una zona profonda del cervello coinvolta nel controllo motorio, nodi `Tian_*-lh` qui sopra), con un danno che degrada dal 100% (putamen posteriore, il piu' colpito) fino al 50% (insula, ai bordi della lesione). E' esattamente il pattern anatomico atteso per uno stroke sottocorticale — un buon segno che il calcolo e' corretto, non solo che "gira senza errori".

## 4. Applicare il risultato alla matrice di connettivita' vera

Ultimo passo: azzerare, nella matrice FC gia' calcolata per questo paziente, tutte le righe e colonne delle zone risultate compromesse (stesso principio di Siegel et al.: se una zona e' distrutta, la sua "connettivita'" con qualsiasi altra zona non ha piu' senso e va rimossa, non lasciata come se fosse un dato valido).

**Punto critico**: l'abbinamento tra la lista di zone compromesse e le righe/colonne della matrice FC deve avvenire **per nome della zona**, mai assumendo che la riga N-esima della matrice corrisponda alla zona N-esima della nostra lista — un disallineamento anche di una sola posizione mischierebbe i risultati di zone diverse senza che nessun errore lo segnali. Per questo verifichiamo esplicitamente che i nomi combacino prima di procedere.

In [5]:
fc_path = glob.glob(f"{DATA_ROOT}/features/{SUBJECT}/func/*{ATLAS_COMBO}*.csv")[0]
fc = pd.read_csv(fc_path, sep="\t", index_col=0)  # la matrice di connettivita' gia' calcolata per questo paziente

# Controllo di sicurezza: se l'ordine dei nomi non combacia esattamente, ci fermiamo qui con un errore
# chiaro, invece di azzerare per sbaglio le zone sbagliate.
assert list(fc.index) == list(node_names), "ordine nodi FC non combacia con l'atlante - mai allineare per posizione"
print(f"Matrice FC caricata: {fc.shape}, allineamento nomi nodo verificato.\n")

fc_masked = fc.copy()  # non modifichiamo mai i dati originali - lavoriamo su una copia
compromised_names = node_names[compromised]  # solo i nomi delle zone da azzerare
fc_masked.loc[compromised_names, :] = 0.0  # azzera tutte le RIGHE delle zone compromesse
fc_masked.loc[:, compromised_names] = 0.0  # azzera tutte le COLONNE delle zone compromesse (la matrice e' simmetrica)

example_node = compromised_names[0]
print(f"Esempio - riga del nodo compromesso '{example_node}', originale vs mascherata:")
print("originale: ", fc.loc[example_node].values[:6])
print("mascherata:", fc_masked.loc[example_node].values[:6])

print(f"\nValori non-zero nella matrice: originale={int((fc.values != 0).sum())}, mascherata={int((fc_masked.values != 0).sum())}")

Matrice FC caricata: (239, 239), allineamento nomi nodo verificato.

Esempio - riga del nodo compromesso '7Networks_LH_SalVentAttn_Ins_1', originale vs mascherata:
originale:  [-0.11554876  0.0087889  -0.10546405 -0.1090772  -0.16777018 -0.16787197]
mascherata: [0. 0. 0. 0. 0. 0.]

Valori non-zero nella matrice: originale=57121, mascherata=53361


## Prossimo passo

Questo e' un **prototipo**, validato per ora su un solo paziente e una sola versione della mappa cerebrale (ce ne sono 12 in totale, con diversi livelli di dettaglio: `Yan{100,200,300,400}TianS{1,2,3}Buckner7N`). Prima di usarlo su tutti i pazienti va:

1. Ripetuto su piu' pazienti e piu' mappe, per essere sicuri che regga in tutti i casi (specie i due casi limite gia' trovati: zone molto piccole, zone completamente lesionate).
2. Trasformato da notebook esplorativo a codice "di produzione" — spostato in `src/features/functional.py`, con test automatici, seguendo lo stesso percorso gia' fatto in passato per l'analisi delle sole lesioni (`notebooks/lesion_analysis.ipynb` -> `src/features/lesion.py`).